# Phase 8 — Evaluation
# Topic 2: RAGAS (Retrieval-Augmented Generation Assessment)

RAGAS is one of the **most frequently asked evaluation frameworks** in GenAI interviews.

Many interviewers ask:

- What is RAGAS?
- Why do we need RAGAS?
- How is it different from DeepEval?
- What metrics does RAGAS provide?
- How do you evaluate a RAG system?

If LangSmith tells you **what happened**, **RAGAS tells you how good your RAG system is**.

---

# 1. What is RAGAS?

## Definition

**RAGAS (Retrieval-Augmented Generation Assessment)** is an open-source framework specifically designed to evaluate **RAG systems**.

It evaluates

- Retrieval quality
- Context quality
- Generated answer quality

without requiring large manually labeled datasets.

---

## Interview Answer

> RAGAS is an evaluation framework specifically designed for Retrieval-Augmented Generation systems. It measures retrieval quality and answer quality using metrics such as Faithfulness, Answer Relevancy, Context Precision, Context Recall, and Context Relevancy, helping developers identify weaknesses in RAG pipelines.

---

# 2. Why Do We Need RAGAS?

Suppose your HR chatbot answers

```text
Employees receive 25 leave days.
```

Correct answer

```text
Employees receive 20 leave days.
```

Without evaluation

```text
Looks okay.
```

But

Is it

- Hallucinated?
- Retrieved wrong document?
- Prompt issue?
- LLM issue?

You don't know.

---

RAGAS tells you

```text
Retriever

↓

Retrieved Wrong Chunk

↓

Faithfulness = Low

↓

Context Recall = Poor
```

Now you know exactly what failed.

---

# 3. Enterprise Architecture

```text
                    User

                     │

                     ▼

                FastAPI

                     │

                     ▼

               LangGraph

                     │

                     ▼

               Retriever

                     │

                     ▼

      OpenSearch / Azure AI Search

                     │

                     ▼

          Bedrock / Azure OpenAI

                     │

                     ▼

                 Response

                     │

                     ▼

                  RAGAS

     ┌──────────┬───────────┬─────────────┐

     ▼          ▼           ▼

 Faithfulness Context Precision Answer Quality
```

---

# AWS + Azure Components

| Layer | AWS | Azure |
|--------|------|--------|
| LLM | Bedrock | Azure OpenAI |
| Retriever | OpenSearch | Azure AI Search |
| Evaluation | RAGAS | RAGAS |
| Observability | LangSmith | LangSmith |

Notice

RAGAS is

Cloud Agnostic.

---

# 4. What Does RAGAS Evaluate?

RAGAS mainly evaluates

```text
Question

↓

Retrieved Context

↓

Generated Answer
```

It analyzes all three.

---

# 5. Input to RAGAS

Example

Question

```text
What is annual leave policy?
```

Retrieved Context

```text
Employees receive 20 annual leave days.
```

Generated Answer

```text
Employees receive 20 leave days.
```

Ground Truth (Optional)

```text
Employees receive 20 annual leave days.
```

RAGAS computes metrics.

---

# 6. RAGAS Metrics

The most important metrics are:

- Faithfulness
- Answer Relevancy
- Context Precision
- Context Recall
- Context Relevancy

Let's understand each one.

---

# 7. Faithfulness ⭐⭐⭐⭐⭐

## Definition

Checks

**Did the LLM answer only using the retrieved context?**

---

Context

```text
Employees receive 20 leave days.
```

Answer

```text
Employees receive 25 leave days.
```

Faithfulness

```text
Low
```

Because

25 never appeared in context.

---

Good Example

Context

```text
20 leave days
```

Answer

```text
20 leave days
```

Faithfulness

```text
High
```

---

Interview Answer

> Faithfulness measures whether the generated answer is supported by the retrieved context. It helps detect hallucinations.

---

# 8. Answer Relevancy

Definition

Does the answer actually answer the user's question?

---

Question

```text
How many leave days?
```

Answer

```text
The company has HR policies.
```

Answer Relevancy

Low

---

Question

```text
How many leave days?
```

Answer

```text
Employees receive 20 annual leave days.
```

High

---

Interview Answer

> Answer Relevancy measures how well the generated answer addresses the user's question.

---

# 9. Context Precision

Definition

How much of the retrieved context is actually useful?

---

Retrieved

```text
Annual Leave

Medical Leave

Travel

Payroll

Insurance
```

Only

```text
Annual Leave
```

needed.

Precision

Low

---

Good Retrieval

```text
Annual Leave
```

Only

Precision

High

---

Interview Answer

> Context Precision measures how relevant the retrieved documents are to the user's question.

---

# 10. Context Recall

Definition

Did retrieval retrieve all necessary information?

---

Question

```text
Explain Leave Policy.
```

Need

```text
Annual Leave

Medical Leave

Carry Forward
```

Retriever only found

```text
Annual Leave
```

Recall

Low

---

Interview Answer

> Context Recall measures whether all relevant information needed to answer the question was successfully retrieved.

---

# 11. Context Relevancy

Definition

Measures how relevant the retrieved context is to the user's question.

---

Question

```text
Leave Policy
```

Retrieved

```text
Payroll
```

Low

---

Question

```text
Leave Policy
```

Retrieved

```text
Leave Policy
```

High

---

# 12. RAGAS Flow

```text
Question

↓

Retriever

↓

Context

↓

LLM

↓

Answer

↓

RAGAS

↓

Scores
```

---

# 13. Installing

```bash
pip install ragas
```

---

# 14. Simple Example

```python
# ==========================================================
# STEP 1 : Import Libraries
# ==========================================================

from datasets import Dataset
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)


# ==========================================================
# STEP 2 : Create Evaluation Dataset
# ==========================================================

dataset = Dataset.from_dict({

    "question": [
        "How many annual leave days are provided?"
    ],

    "answer": [
        "Employees receive 20 annual leave days."
    ],

    "contexts": [[
        "Employees receive 20 annual leave days. Unused leave can be carried forward."
    ]],

    "ground_truth": [
        "Employees receive 20 annual leave days."
    ]
})


# ==========================================================
# STEP 3 : Evaluate
# ==========================================================

results = evaluate(

    dataset,

    metrics=[

        faithfulness,

        answer_relevancy,

        context_precision,

        context_recall
    ]
)


# ==========================================================
# STEP 4 : Display Results
# ==========================================================

print(results)
```

---

# Sample Output

```text
Faithfulness        : 0.98

Answer Relevancy    : 0.95

Context Precision   : 0.91

Context Recall      : 0.97
```

Scores range from

```text
0 → Poor

1 → Excellent
```

---

# 15. Production Architecture

```text
User

↓

FastAPI

↓

LangGraph

↓

Retriever

↓

OpenSearch

↓

Bedrock

↓

Answer

↓

LangSmith

↓

RAGAS

↓

Dashboard
```

---

# 16. Advantages

✅ Designed specifically for RAG

✅ No large labeled dataset required

✅ Detects hallucinations

✅ Evaluates retrieval quality

✅ Easy integration

---

# 17. Disadvantages

❌ Additional LLM calls

❌ Increased evaluation cost

❌ Longer evaluation time

❌ Primarily focused on RAG, not general agent behavior

---

# 18. Best Practices

✅ Evaluate after every prompt update.

✅ Evaluate after changing chunk size.

✅ Evaluate after changing embedding models.

✅ Track scores over time.

✅ Store evaluation history.

---

# 19. Common Mistakes

❌ Only checking answer correctness.

❌ Ignoring retrieval quality.

❌ Evaluating only a few examples.

❌ Never monitoring score degradation after deployment.

---

# 20. RAGAS vs LangSmith

| LangSmith | RAGAS |
|------------|--------|
| Observability | Evaluation |
| Trace execution | Score quality |
| Debug prompts | Measure retrieval |
| Token usage | Faithfulness |
| Latency | Context metrics |

Use both together.

---

# 21. RAGAS vs DeepEval

| RAGAS | DeepEval |
|--------|----------|
| RAG focused | General LLM evaluation |
| Retrieval metrics | Broader evaluation metrics |
| Context quality | Agents, prompts, safety, hallucination, etc. |
| Best for RAG | Best for end-to-end AI systems |

---

# 22. Real Enterprise Example

### HR Assistant

Question

```text
What is maternity leave policy?
```

Pipeline

```text
Hybrid Search

↓

Retrieved Context

↓

Claude

↓

Answer

↓

RAGAS
```

Scores

```text
Faithfulness : 0.97

Answer Relevancy : 0.95

Context Recall : 0.94
```

If Context Recall drops to **0.60**, it suggests the retriever is missing important HR policy sections.

---

### Healthcare Assistant

Question

```text
What are the contraindications for Metformin?
```

RAGAS can reveal whether:

- The retrieved clinical guideline contained the required information.
- The answer stayed grounded in the retrieved context.
- Important contraindications were omitted.

---

# 23. Common Interview Questions

### Q1. What is RAGAS?

A framework for evaluating Retrieval-Augmented Generation systems.

---

### Q2. Does RAGAS evaluate retrieval?

Yes.

It evaluates:

- Context Precision
- Context Recall
- Context Relevancy

---

### Q3. Does RAGAS detect hallucinations?

Yes.

Faithfulness is one of its key metrics for identifying answers that are not supported by the retrieved context.

---

### Q4. Does RAGAS require ground truth?

Not always.

Some metrics (like Faithfulness) do not require a reference answer, while others (such as answer correctness in certain evaluation setups) benefit from ground truth.

---

### Q5. Can RAGAS work with AWS Bedrock?

Yes.

It is model-agnostic and works with Bedrock, Azure OpenAI, OpenAI, Anthropic APIs, and other LLM providers.

---

# 24. Complete Enterprise Evaluation Pipeline

```text
User
      │
      ▼
FastAPI
      │
      ▼
LangGraph
      │
      ▼
Query Rewriting
      │
      ▼
Hybrid Search
      │
      ▼
Retriever
      │
      ▼
Bedrock / Azure OpenAI
      │
      ▼
Answer
      │
      ├────────────► LangSmith
      │                │
      │                ├── Traces
      │                ├── Tokens
      │                └── Latency
      │
      ▼
RAGAS
      │
      ├── Faithfulness
      ├── Answer Relevancy
      ├── Context Precision
      ├── Context Recall
      └── Context Relevancy
```

---

# 25. EPAM Senior Answer (3–4 Minutes)

> "RAGAS is an evaluation framework specifically designed for Retrieval-Augmented Generation systems. Unlike observability tools such as LangSmith, which trace execution, RAGAS measures the quality of retrieval and generated responses. It evaluates metrics such as Faithfulness, which checks whether the answer is supported by the retrieved context; Answer Relevancy, which measures whether the response addresses the user's question; Context Precision, which measures how relevant the retrieved documents are; and Context Recall, which measures whether all required information was retrieved. In a production enterprise AI application, I typically use LangSmith to trace the entire LangGraph workflow—including retrieval, prompts, and model calls—and RAGAS to quantify retrieval quality. On AWS, this integrates well with Bedrock and OpenSearch, while on Azure it works with Azure OpenAI and Azure AI Search. Using both tools together provides complete observability and evaluation for production-grade RAG systems."